In [21]:
import requests
import os
import random
import json
import datetime
from urllib.parse import unquote, quote, urlparse, unquote

def generate_random_duration():
    # Durasi dalam detik, antara 30 detik dan 5 menit (300 detik)
    duration_seconds = random.randint(30, 300)
    minutes = duration_seconds // 60
    seconds = duration_seconds % 60
    return f"{minutes:02}:{seconds:02}"

def escape_sql_string(value):
    # Escape single quotes by doubling them
    return value.replace("'", "''")

def escape_ampersand(url):
    # Mengencode ampersand menjadi %26
    if "&" in url:
        return url.replace("&", "%26")
    if " " in url:
        return url.replace(" ", "%20")
    if "⁇" in url:
        return url.replace("⁇", "%E2%81%87")
    if "’" in url:
        return url.replace("’", "%E2%80%99")
    else:
        return url

def escape_url(url):
    # Mengencode karakter khusus lainnya menggunakan urllib
    return quote(url, safe=":/")

def get_artist_from_title(title, list_title_artist):
    """Fungsi untuk mencocokkan judul dan menemukan artis yang sesuai."""
    for item in list_title_artist:
        # Pisahkan berdasarkan ' --- '
        song_title, artist = item.split(' --- ')
        if title.lower() == song_title.lower():  # Cocokkan tanpa memperhatikan huruf besar/kecil
            return artist
    return "Unknown Artist"  # Return "Unknown Artist" string jika tidak ditemukan

def get_match_title_from_name(title, list_index_title):
    """Fungsi untuk mencocokkan index dan menemukan judul yang sesuai."""
    for item in list_index_title:
        # Pisahkan berdasarkan ' --- '
        song_title, artist = item.split(' --- ')
        if title.lower() == song_title.lower():  # Cocokkan tanpa memperhatikan huruf besar/kecil
            return artist
    return "Unknown Title"  # Return "Unknown Title" string jika tidak ditemukan

def fetch_gitrepo_files(repo_url):
    """
    Example URL Folder:
    Github: https://github.com/cybeat-music/mitsuboshi-colors/tree/main/COLORS%20POWER
    Bitbucker: https://bitbucket.org/sibeux/arabasta-boom/src/main/Lost_Eve/
    Codeberg: https://codeberg.org/sibeux/regalia-freeze/src/branch/main/KONTINUUM
    Gitlab: https://gitlab.com/habiqi/regarden-academy/-/tree/main/Watch%20me?ref_type=heads
    """

    # Ekstrak informasi pemilik, repository, dan path folder dari URL
    parts = repo_url.split("/")
    owner = parts[3]
    repo = parts[4]
    branch = parts[6]
    folder_path = "/".join(parts[7:])
    music_path = []

    # Daftar pasangan nama lagu dan artis
    list_title_artist = []
    # Daftar pasangan nama lagu dan index
    list_index_title = []

    list_cover = []
    category = ""
    artist = ""
    album_artist = ""
    album = ''

    # Untuk mencari nama file
    file_name_key = ""
    # Untuk cek jenis file =  direktori
    file_dir_value = ""
    # Untuk cek jenis file = file
    file_type_value = ""

    is_dynamic_artist = False
    is_dynamic_title = False
    disc = 0

    folder_api_url = ""

    if "github.com/" in repo_url:
        folder_api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{folder_path}"
    elif "https://bitbucket.org/" in repo_url:
        folder_api_url = f"https://api.bitbucket.org/2.0/repositories/{owner}/{repo}/src/{branch}/{folder_path}"
    elif "https://codeberg.org"  in repo_url:
        # Khusus untuk Codeberg & Gitlab, pakai index 8 karena link-nya lebih panjang
        folder_path = "/".join(parts[8:])
        folder_api_url = f"https://codeberg.org/api/v1/repos/{owner}/{repo}/contents/{folder_path}"
    elif "https://gitlab.com/" in repo_url:
        folder_path = parts[8].split("?")[0]  # Mengambil path sebelum query string
        folder_api_url = f"https://gitlab.com/api/v4/projects/{owner}%2F{repo}/repository/tree?path={folder_path}&ref=main"

    folder_response = requests.get(folder_api_url)

    if folder_response.status_code == 200:
        files = folder_response.json()
        if "github.com/" in repo_url:
            file_name_key = "name"
            file_dir_value = "dir"
            file_type_value = "file"
        elif "https://bitbucket.org/" in repo_url:
            file_name_key = "path"
            file_dir_value = "commit_directory"
            file_type_value = "commit_file"
            files = files.get("values", [])
        elif "https://codeberg.org" in repo_url:
            file_name_key = "name"
            file_dir_value = "dir"
            file_type_value = "file"
        elif "https://gitlab.com/" in repo_url:
            file_name_key = "name"
            file_dir_value = "tree"
            file_type_value = "blob"
        for file in files:
            if file["type"] == file_dir_value:
                file_name_without_ext = os.path.splitext(file[file_name_key])[0]
                if "https://bitbucket.org/" in repo_url:
                    file_name_without_ext = file[file_name_key].split("/")[-1]
                if file_name_without_ext == "flac":
                    music_path.append(file["path"])
                if file_name_without_ext.isdigit():
                    music_path.append(file["path"])
            if '.here.txt' in file[file_name_key]:
                txt_url = ""
                if "github.com/" in repo_url:
                    txt_url = file["download_url"]
                elif "https://bitbucket.org/" in repo_url:
                    txt_url = file['links']['self']['href']
                elif "https://codeberg.org" in repo_url:
                    txt_url = file['download_url']
                elif "https://gitlab.com/" in repo_url:
                    branch = parts[7]
                    txt_url = f"https://gitlab.com/{owner}/{repo}/-/raw/{branch}/{folder_path}/.here.txt"
                
                response = requests.get(txt_url)
                # Secara eksplisit set encoding ke 'utf-8'
                response.encoding = 'utf-8'
                text_content = json.loads(response.text)
                category = text_content["category"]
                album = text_content['title']
                album = album.replace("'", "`")
                artist = text_content["author"]
                album_artist = text_content["author"]
                if text_content['disc'] != '0':
                    disc = int(text_content['disc']) - 1
                    if (text_content['1']) != []:
                        is_dynamic_artist = True
                        for i in range(disc+1):
                            list_title_artist.append(
                                text_content[str(i+1)]
                            )
                elif text_content['1'] != []:
                    is_dynamic_artist = True
                    list_title_artist.append(text_content["1"])
                if text_content['music_name_index']['1'] != []:
                    is_dynamic_title = True
                    for i in range(disc+1):
                        list_index_title.append(
                            text_content['music_name_index'][str(i+1)]
                        )
            if file['type'] == file_type_value:
                ext = os.path.splitext(file[file_name_key])[1].lower()
                if ext in ['.jpg', '.jpeg', '.png']:
                    image_url = ""
                    if "github.com/" in repo_url:
                        image_url = file["download_url"]
                    elif "https://bitbucket.org/" in repo_url:
                        image_url = file['links']['self']['href']
                    elif "https://codeberg.org" in repo_url:
                        image_url = file['download_url']
                    elif "https://gitlab.com/" in repo_url:
                        image_url = f"https://gitlab.com/{owner}/{repo}/-/raw/{branch}/{folder_path}/{file[file_name_key]}"
                    list_cover.append(image_url)

    # Starting timestamp for date_added
    base_time = datetime.datetime.now().replace(microsecond=0)

    # Base SQL INSERT statement template
    insert_statement_template = "INSERT INTO music (id_music, category, link_gdrive, title, artist, album, time, cover, favorite, date_added) VALUES \n"
    playlist_insert_statement_template = "INSERT INTO `playlist` (`uid`, `name`, `image`, `type`, `author`, `pin`, `date_pin`, `date`, `editable`) VALUES \n"

    # Parameters for the static fields
    # 1 = IndoPride
    # 2 = 日本の歌
    # 3 = Jowo Mletre
    # 4 = Worldwide
    favorite = "0"

    # Collecting each row of values
    values = []
    playlist_value = []

    for x in range(disc+1):
        # Buat URL API untuk folder
        api_url = ""
        if "github.com/" in repo_url:
            api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{music_path[x]}"
        elif "https://bitbucket.org/" in repo_url:
            api_url = f"https://api.bitbucket.org/2.0/repositories/{owner}/{repo}/src/{branch}/{music_path[x]}?pagelen=100"
        elif "https://codeberg.org" in repo_url:
            api_url = f"https://codeberg.org/api/v1/repos/{owner}/{repo}/contents/{music_path[x]}"
        elif "https://gitlab.com/" in repo_url:
            api_url = f"https://gitlab.com/api/v4/projects/{owner}%2F{repo}/repository/tree?path={music_path[x]}&ref=main&per_page=100&page=1"

        # Lakukan request ke API Git Repository
        response = requests.get(api_url)

        if response.status_code == 200:
            files = response.json()
            if "https://bitbucket.org/" in repo_url:
                files = files.get("values", [])
            index = 0
            album_disc = f"{album} Disc {x+1}"
            album_name = ''
            if disc > 0:
                album_name = album_disc
            else:
                album_name = album
            for file in files:
                if file["type"] == file_type_value:
                    # Hapus ekstensi file untuk menampilkan nama file tanpa ekstensi
                    file_name_without_ext = os.path.splitext(
                        file[file_name_key])[0]
                    if "https://bitbucket.org/" in repo_url:
                        file_name_without_ext = file_name_without_ext.split("/")[-1]
                    title_song = ""

                    # cari artist berdasarkan nama file
                    if is_dynamic_artist:
                        artist = get_artist_from_title(
                            file_name_without_ext, list_title_artist[x])
                        
                    # Cari judul berdasarkan index
                    if is_dynamic_title:
                        title_song = get_match_title_from_name(file_name_without_ext, list_index_title[x])
                        title_song = title_song.replace("\'", "\\'")

                    date_added = base_time + datetime.timedelta(seconds=index)
                    date_added_str = date_added.strftime("%Y-%m-%d %H:%M:%S")

                    index += 1
                    duration = generate_random_duration()

                    # Escape special characters in the values
                    artist = escape_sql_string(artist)
                    album = escape_sql_string(album)

                    # mendapatkan link git repo yang bisa di-stream
                    download_music_link = ""
                    if "github.com/" in repo_url:
                        download_music_link = file['download_url']
                    elif "https://bitbucket.org/" in repo_url:
                        download_music_link = file['links']['self']['href']
                    elif "https://codeberg.org" in repo_url:
                        download_music_link = file['download_url']
                    elif "https://gitlab.com/" in repo_url:
                        download_music_link = f"https://gitlab.com/{owner}/{repo}/-/raw/{branch}/{folder_path}/flac/{file[file_name_key]}"

                    music_link = download_music_link

                    if "github.com/" in repo_url:
                        # mendapatkan link file yang versi encode
                        name_file_link_html = file['html_url']
                        # mengambil nama file dan ekstensi dari link html
                        name_link_encoded = name_file_link_html.split('/')[-1]
                        # memasukkan ke dalam link stream dengan nama file encode
                        music_link = download_music_link.rsplit(
                            '/', 1)[0] + '/' + name_link_encoded
                    # Encode ampersand
                    music_link = escape_ampersand(music_link)
                    # Encode other special characters
                    music_link = escape_url(music_link)
                    # Unquote untuk menghapus encoding berlebih
                    music_link = unquote(music_link)
                    # Khusus untuk Gitlab, encode ampersand lagi
                    if "https://gitlab.com/" in repo_url:
                        music_link = escape_ampersand(music_link)

                    cover_link = list_cover[(index-1) % len(list_cover)]
                    cover_link = escape_sql_string(cover_link)
                    if "github.com/" in repo_url:
                        cover_link = cover_link.replace(
                            "https://github.com/",
                            "https://raw.githubusercontent.com/"
                        ).replace(
                            "/blob/",
                            "/refs/heads/"
                        )
                    date_added_str = escape_sql_string(date_added_str)

                    final_title = ""
                    if is_dynamic_title == True:
                        final_title = title_song
                    else:
                        final_title = file_name_without_ext

                    values.append(
                        f"(NULL, '{category}', '{music_link}', '{final_title}', '{artist}', '{album_name}', '{duration}', '{cover_link}', '{favorite}', '{date_added_str}')"
                    )
                    if x == 0 and index == 1:
                        playlist_value.append(
                            f"(NULL, '{album}', '{cover_link}', 'album', '{album_artist}', 'false', NULL, NOW(), 'false')"
                        )
        else:
            print("Error:", response.status_code, response.json())

    # Joining all values into the final SQL INSERT statement
    insert_statement = insert_statement_template + \
            ",\n".join(values) + ";"
    playlist_insert_statement = playlist_insert_statement_template + \
            ",\n".join(playlist_value) + ";"
    print(insert_statement)
    print(playlist_insert_statement)


# Place URL here
repo_url = "https://gitlab.com/habiqi/sibeux/-/tree/main/Fantasy%20Life%20OST?ref_type=heads"

fetch_gitrepo_files(repo_url)

INSERT INTO music (id_music, category, link_gdrive, title, artist, album, time, cover, favorite, date_added) VALUES 
(NULL, '2', 'https://gitlab.com/habiqi/sibeux/-/raw/main/Fantasy%20Life%20OST/flac/Ally%20Made.flac', 'Ally Made', 'Nobuo Uematsu', 'Fantasy Life Original Soundtrack', '02:59', 'https://gitlab.com/habiqi/sibeux/-/raw/main/Fantasy%20Life%20OST/00 Front.jpg', '0', '2025-06-16 12:14:42'),
(NULL, '2', 'https://gitlab.com/habiqi/sibeux/-/raw/main/Fantasy%20Life%20OST/flac/Ancient%20Ruins.flac', 'Ancient Ruins', 'Nobuo Uematsu', 'Fantasy Life Original Soundtrack', '00:52', 'https://gitlab.com/habiqi/sibeux/-/raw/main/Fantasy%20Life%20OST/00 Front.jpg', '0', '2025-06-16 12:14:43'),
(NULL, '2', 'https://gitlab.com/habiqi/sibeux/-/raw/main/Fantasy%20Life%20OST/flac/Aridian%20Desert%20Theme.flac', 'Aridian Desert Theme', 'Nobuo Uematsu', 'Fantasy Life Original Soundtrack', '03:39', 'https://gitlab.com/habiqi/sibeux/-/raw/main/Fantasy%20Life%20OST/00 Front.jpg', '0', '2025-06-16 12